# 🛒 Exploratory Data Analysis on E-Commerce Sales Data

## Overview
This project analyzes an e-commerce sales dataset to uncover customer purchasing behavior, product performance, seasonal revenue patterns, and regional trends.

**Dataset:** [E-Commerce Data – Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data)  
**File:** `data.csv`

## Importing all the Important Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully ✅')

## Loading the Dataset

In [ ]:
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns ✅')

## Checking First Five Rows of Dataset

In [ ]:
df.head()

## Checking Last Five Rows of Dataset

In [ ]:
df.tail()

## Total Number of Rows and Columns Present in Dataset

In [ ]:
df.shape

## Checking Data Types of Each Column

In [ ]:
df.dtypes

## An Overview on Dataset

In [ ]:
df.info()

# Performing Data Cleaning

## Checking Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
# % of missing values
(df.isnull().sum() / df.shape[0]) * 100

In [ ]:
# Total missing %
(df.isnull().sum().sum()) / (df.shape[0] * df.shape[1]) * 100

## Handling Missing Values

In [ ]:
# Drop rows where CustomerID is missing (can't analyze without customer)
df.dropna(subset=['CustomerID'], inplace=True)

# Fill missing Description with 'Unknown'
df['Description'].fillna('Unknown', inplace=True)

print('Missing values handled ✅')
df.isnull().sum()

## Checking for Duplicates

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print(f'After dropping duplicates: {df.shape}')

## Handle Negative Quantities and Prices (Returns/Cancellations)

In [ ]:
# Remove cancelled orders (InvoiceNo starting with 'C') and negative quantities
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
print(f'After removing returns/cancellations: {df.shape}')

## Data Type Conversion & Feature Engineering

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Extract time features
df['Year']   = df['InvoiceDate'].dt.year
df['Month']  = df['InvoiceDate'].dt.month
df['Day']    = df['InvoiceDate'].dt.day_name()
df['Hour']   = df['InvoiceDate'].dt.hour

# Calculate Revenue per line
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print('Feature engineering done ✅')
df[['InvoiceDate','Year','Month','Day','Hour','Revenue']].head()

## Statistical Summary

In [ ]:
df[['Quantity','UnitPrice','Revenue']].describe()

In [ ]:
df.describe(include='object')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df[['Quantity','UnitPrice','Revenue']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation — Quantity, Price, Revenue', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Performing Exploratory Data Analysis

## i) Univariate Analysis

In [ ]:
# --- Monthly Revenue Trend ---
monthly_rev = df.groupby(['Year','Month'])['Revenue'].sum().reset_index()
monthly_rev['period'] = monthly_rev['Year'].astype(str) + '-' + monthly_rev['Month'].apply(lambda x: f'{x:02d}')
monthly_rev = monthly_rev.sort_values('period')

plt.figure(figsize=(12, 5))
plt.bar(monthly_rev['period'], monthly_rev['Revenue'] / 1000,
        color='#2563EB', edgecolor='white')
plt.xlabel('Month')
plt.ylabel('Revenue (£ Thousands)')
plt.title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.tight_layout()
plt.show()
print('Revenue peaks sharply in November — Holiday/Christmas shopping season')

In [ ]:
# --- Revenue by Day of Week ---
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_rev = df.groupby('Day')['Revenue'].sum().reindex(day_order)

plt.figure(figsize=(9, 5))
bars = plt.bar(day_rev.index, day_rev.values / 1000, color='#2563EB', edgecolor='white')
plt.xlabel('Day of Week')
plt.ylabel('Revenue (£ Thousands)')
plt.title('Revenue by Day of Week', fontsize=13, fontweight='bold')
for bar, val in zip(bars, day_rev.values / 1000):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'£{val:.0f}K', ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print('Thursday generates the highest revenue; Sunday has zero orders (store closed)')

In [ ]:
# --- Orders by Hour of Day ---
hour_counts = df.groupby('Hour')['InvoiceNo'].nunique()
plt.figure(figsize=(10, 5))
plt.plot(hour_counts.index, hour_counts.values, marker='o',
         color='#2563EB', linewidth=2.5)
plt.fill_between(hour_counts.index, hour_counts.values, alpha=0.15, color='#2563EB')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Orders')
plt.title('Order Volume by Hour of Day', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Peak ordering hours are 10am–12pm (mid-morning)')

## ii) Bivariate Analysis

In [ ]:
# --- Top 10 Countries by Revenue (ex UK) ---
country_rev = df[df['Country'] != 'United Kingdom'].groupby('Country')['Revenue'].sum().nlargest(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(country_rev.index[::-1], country_rev.values[::-1] / 1000, color='#1D4ED8')
plt.xlabel('Revenue (£ Thousands)')
plt.title('Top 10 Countries by Revenue (Excluding UK)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, country_rev.values[::-1] / 1000):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'£{val:.0f}K', va='center', fontsize=8)
plt.tight_layout()
plt.show()
print('Netherlands and Ireland are the top international markets')

In [ ]:
# --- Top 10 Best-Selling Products by Quantity ---
top_products_qty = df.groupby('Description')['Quantity'].sum().nlargest(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(top_products_qty.index[::-1], top_products_qty.values[::-1],
                color='#3B82F6')
plt.xlabel('Total Quantity Sold')
plt.title('Top 10 Best-Selling Products by Quantity', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top_products_qty.values[::-1]):
    plt.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             str(int(val)), va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 10 Products by Revenue ---
top_products_rev = df.groupby('Description')['Revenue'].sum().nlargest(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(top_products_rev.index[::-1], top_products_rev.values[::-1] / 1000,
                color='#1D4ED8')
plt.xlabel('Total Revenue (£ Thousands)')
plt.title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top_products_rev.values[::-1] / 1000):
    plt.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'£{val:.1f}K', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Revenue Distribution by Country (Top 10 incl UK) ---
top_countries_all = df.groupby('Country')['Revenue'].sum().nlargest(10)
plt.figure(figsize=(7, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_countries_all)))
plt.pie(top_countries_all, labels=top_countries_all.index,
        autopct='%1.1f%%', colors=colors[::-1],
        startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
plt.title('Revenue Share by Country (Top 10)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('UK accounts for ~85% of total revenue — highly concentrated market')

## iii) Multivariate Analysis

In [ ]:
# --- Monthly Revenue by Country (Top 5) ---
top5_countries = df.groupby('Country')['Revenue'].sum().nlargest(5).index.tolist()
df_top5 = df[df['Country'].isin(top5_countries)]
monthly_country = df_top5.groupby(['Month', 'Country'])['Revenue'].sum().unstack(fill_value=0)

monthly_country.plot(figsize=(12, 5), colormap='tab10', linewidth=2, marker='o', markersize=4)
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.title('Monthly Revenue by Top 5 Countries', fontsize=13, fontweight='bold')
plt.legend(title='Country')
plt.tight_layout()
plt.show()

In [ ]:
# --- RFM Analysis Setup ---
# RFM = Recency, Frequency, Monetary
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('Revenue',      'sum')
).reset_index()

print('RFM Table (first 5 rows):')
print(rfm.head())
print(f'\nTotal unique customers: {len(rfm):,}')

In [ ]:
# --- RFM Score Distribution ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Distribution — Customer Analysis', fontsize=14, fontweight='bold')

axes[0].hist(rfm['Recency'],   bins=40, color='#EF4444', edgecolor='white')
axes[0].set_title('Recency (days since last purchase)')
axes[0].set_xlabel('Days')

axes[1].hist(rfm['Frequency'], bins=40, color='#3B82F6', edgecolor='white')
axes[1].set_title('Frequency (number of orders)')
axes[1].set_xlabel('Orders')

axes[2].hist(rfm['Monetary'],  bins=40, color='#10B981', edgecolor='white')
axes[2].set_title('Monetary (total spend £)')
axes[2].set_xlabel('£ Spent')

plt.tight_layout()
plt.show()

In [ ]:
# --- Customer Segmentation by Purchase Frequency ---
rfm['Segment'] = pd.cut(rfm['Frequency'],
                         bins=[0, 1, 3, 10, rfm['Frequency'].max()],
                         labels=['One-Time', 'Occasional', 'Regular', 'Loyal'])

seg_counts = rfm['Segment'].value_counts()
plt.figure(figsize=(7, 5))
colors_seg = ['#EF4444','#F97316','#3B82F6','#10B981']
plt.pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
        colors=colors_seg, startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Customer Segments by Purchase Frequency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(seg_counts)

## Key Insights & Findings

### 1. Revenue Seasonality
- **November** is the peak revenue month — Christmas/holiday shopping.
- **Thursday** is the top revenue day; Sundays show no activity.

### 2. Geography
- **UK** contributes ~85% of all revenue.
- **Netherlands, Ireland, Germany** are the top international markets.

### 3. Products
- A small number of **gift/decorative items** drive the majority of quantity sold.
- High-revenue products differ from high-quantity products.

### 4. Customer Behavior
- **Peak ordering time** is 10am–12pm.
- RFM shows majority of customers are **one-time or occasional buyers** — retention opportunity.

### 5. RFM Segmentation
- Only a small % are **Loyal customers** — high lifetime value target segment.

## Conclusion
This EDA of the e-commerce dataset reveals strong seasonal patterns, UK market dominance, and clear customer segmentation opportunities. The RFM framework provides a foundation for targeted marketing.

---
### 📌 Next Steps
- Build a **Customer Lifetime Value (CLV)** model.
- Implement **Market Basket Analysis** (Association Rules) for cross-selling.
- Develop a **churn prediction model** for at-risk customers.

**📊 Tools Used:** Python, Pandas, Matplotlib, Seaborn, Jupyter Notebook

---
### -----------------------------------------THANKYOU---------------------------------